# Dataset-level correlation analysis — corrected version

Counterfactual metrics are aggregated across the seven methods within each dataset before cross-dataset Spearman analysis. Benjamini–Hochberg correction and bootstrap confidence intervals are preserved from the final analysis.


In [ ]:
from pathlib import Path

def find_project_root(start=None):
    """Locate the repository root from the current working directory."""
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "Data").is_dir() and (candidate / "Results").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate CFE_for_MTS_Forecasting_Framework. "
        "Run this notebook from somewhere inside the cloned repository."
    )

PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)


from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

RANDOM_SEED = 42


In [ ]:
# 1. PATHS
BASE_DIR = PROJECT_ROOT / "Results" / "EDBT-results-public-datasets" / "All"
PIEZO_FORECAST_FILE = BASE_DIR / "forecast_metrics.csv"
PUBLIC_FORECAST_FILE = BASE_DIR / "best_models_test_re_evaluation.csv"
PIEZO_CF_FILE = BASE_DIR / "counterfactual_summary_by_dataset_method_stats_piezo.csv"
PUBLIC_CF_FILE = BASE_DIR / "counterfactual_summary_by_dataset_method_stats_public.csv"
OUTPUT_DIR = BASE_DIR / "meta_analysis_corrected"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Reading files from:", BASE_DIR)
print("Saving outputs to:", OUTPUT_DIR)


In [ ]:
# 2. LOAD FORECAST R²
piezo_forecast = pd.read_csv(PIEZO_FORECAST_FILE)
public_forecast = pd.read_csv(PUBLIC_FORECAST_FILE)

if not {"dataset", "test_r2_full"}.issubset(piezo_forecast.columns):
    raise ValueError("Piezo forecast file must contain dataset and test_r2_full")
if not {"dataset", "test_r2"}.issubset(public_forecast.columns):
    raise ValueError("Public forecast file must contain dataset and test_r2")

piezo_r2 = piezo_forecast[["dataset", "test_r2_full"]].rename(columns={"test_r2_full":"forecast_r2"})
public_r2 = public_forecast[["dataset", "test_r2"]].rename(columns={"test_r2":"forecast_r2"})
forecast_r2 = pd.concat([piezo_r2, public_r2], ignore_index=True)

if forecast_r2["dataset"].duplicated().any():
    dup = forecast_r2.loc[forecast_r2["dataset"].duplicated(keep=False), "dataset"].tolist()
    raise ValueError(f"Duplicate forecast rows found for dataset(s): {dup}")
if forecast_r2["forecast_r2"].isna().any():
    raise ValueError("Missing forecast R² values detected")

print("Forecast datasets:", forecast_r2["dataset"].nunique())
print(forecast_r2.sort_values("dataset").to_string(index=False))


In [ ]:
# 3. LOAD COUNTERFACTUAL DATASET × METHOD RESULTS
piezo_cf = pd.read_csv(PIEZO_CF_FILE)
public_cf = pd.read_csv(PUBLIC_CF_FILE)
cf = pd.concat([piezo_cf, public_cf], ignore_index=True)

metric_columns = {
    "validity_ratio_mean": "validity_ratio",
    "stepwise_validity_auc_mean": "prefix_validity",
    "consecutive_valid_steps_mean": "prefix_valid_steps",
    "max_consecutive_valid_steps_mean": "max_consecutive_valid_steps",
    "distance_to_band_mean": "distance_to_band",
    "proximity_l2_scaled_mean": "proximity_l2_scaled",
    "proximity_l1_scaled_mean": "proximity_l1_scaled",
    "sparsity_count_mean": "sparsity_count",
    "sparsity_ratio_mean": "sparsity_ratio",
    "runtime_sec_mean": "runtime_sec",
}

required_cf_cols = {"dataset", "method"} | set(metric_columns.keys())
missing_cf = sorted(required_cf_cols - set(cf.columns))
if missing_cf:
    raise ValueError(f"Missing counterfactual columns: {missing_cf}")

cf_small = cf[["dataset", "method"] + list(metric_columns.keys())].rename(columns=metric_columns)

dups = cf_small.duplicated(["dataset", "method"], keep=False)
if dups.any():
    raise ValueError("Duplicate dataset × method rows detected:\n" + cf_small.loc[dups, ["dataset","method"]].to_string(index=False))

print("CF datasets:", cf_small["dataset"].nunique())
print("CF methods:", cf_small["method"].nunique())
print("CF rows:", len(cf_small))
print("\nMethods per dataset:")
print(cf_small.groupby("dataset")["method"].nunique().to_string())


In [ ]:
# 4. MERGE R² WITH COUNTERFACTUAL RESULTS + STRICT CHECKS
merged = cf_small.merge(forecast_r2, on="dataset", how="left", validate="many_to_one")
if merged["forecast_r2"].isna().any():
    missing_ds = merged.loc[merged["forecast_r2"].isna(), "dataset"].unique().tolist()
    raise ValueError(f"No R² found for dataset(s): {missing_ds}")

n_datasets = merged["dataset"].nunique()
n_methods = merged["method"].nunique()
print(f"Merged rows: {len(merged)}")
print(f"Datasets: {n_datasets}")
print(f"Methods: {n_methods}")

if n_datasets != 21:
    warnings.warn(f"Expected 21 datasets, found {n_datasets}")
if n_methods != 7:
    warnings.warn(f"Expected 7 counterfactual variants, found {n_methods}")
if len(merged) != n_datasets * n_methods:
    raise ValueError("Incomplete dataset × method grid")

merged.to_csv(OUTPUT_DIR / "r2_counterfactual_merged_dataset_method.csv", index=False)


In [ ]:
# 5. PRIMARY CROSS-DATASET METRICS
MAIN_METRICS = {
    "validity_ratio": "Validity",
    "prefix_validity": "Prefix validity",
    "max_consecutive_valid_steps": "Max valid steps",
    "proximity_l2_scaled": "Scaled L2 proximity",
    "proximity_l1_scaled": "Scaled L1 proximity",
    "sparsity_ratio": "Sparsity ratio",
    "runtime_sec": "Runtime",
}
metric_keys = list(MAIN_METRICS.keys())


## Dataset-level aggregation

The main question is dataset-level: **when the forecaster is better on a dataset, are the counterfactual explanations also generally better on that dataset?**

The seven methods are aggregated within each dataset. The **mean across methods** is primary; the **median across methods** is a robustness check. This leaves one observation per dataset.


In [ ]:
# 6. CREATE ONE OBSERVATION PER DATASET
dataset_mean = merged.groupby("dataset", as_index=False).agg(
    forecast_r2=("forecast_r2", "first"),
    **{metric: (metric, "mean") for metric in metric_keys}
)

dataset_median = merged.groupby("dataset", as_index=False).agg(
    forecast_r2=("forecast_r2", "first"),
    **{metric: (metric, "median") for metric in metric_keys}
)

assert len(dataset_mean) == n_datasets
assert len(dataset_median) == n_datasets

dataset_mean.to_csv(OUTPUT_DIR / "dataset_level_mean_cf_metrics.csv", index=False)
dataset_median.to_csv(OUTPUT_DIR / "dataset_level_median_cf_metrics.csv", index=False)

print("Primary dataset-level table (mean across methods):")
print(dataset_mean.round(4).to_string(index=False))


In [ ]:
# 7. BENJAMINI-HOCHBERG FDR CORRECTION
def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adjusted_ranked = ranked * m / np.arange(1, m + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted_ranked = np.clip(adjusted_ranked, 0, 1)
    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adjusted_ranked
    return adjusted


In [ ]:
# 8. BOOTSTRAP 95% CI FOR SPEARMAN rho
def bootstrap_spearman_ci(x, y, n_boot=5000, ci=0.95, seed=RANDOM_SEED):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    local_rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        idx = local_rng.integers(0, n, size=n)
        xb, yb = x[idx], y[idx]
        if np.unique(xb).size < 2 or np.unique(yb).size < 2:
            continue
        rho = spearmanr(xb, yb).statistic
        if np.isfinite(rho):
            values.append(rho)
    alpha = (1-ci)/2
    return np.quantile(values, alpha), np.quantile(values, 1-alpha)


In [ ]:
# 9. PRIMARY ANALYSIS: SPEARMAN R² VS EACH METHOD-AVERAGED CF METRIC
rows = []
for metric, label in MAIN_METRICS.items():
    x = dataset_mean["forecast_r2"].to_numpy()
    y = dataset_mean[metric].to_numpy()
    s = spearmanr(x, y)
    p = pearsonr(x, y)
    ci_low, ci_high = bootstrap_spearman_ci(x, y, n_boot=5000, ci=0.95, seed=RANDOM_SEED)
    rows.append({
        "metric": metric,
        "metric_label": label,
        "n_datasets": len(dataset_mean),
        "spearman_rho": s.statistic,
        "spearman_p_raw": s.pvalue,
        "spearman_ci95_low": ci_low,
        "spearman_ci95_high": ci_high,
        "pearson_r": p.statistic,
        "pearson_p_raw": p.pvalue,
    })

dataset_corr = pd.DataFrame(rows)
dataset_corr["spearman_p_fdr_bh"] = benjamini_hochberg(dataset_corr["spearman_p_raw"].to_numpy())
dataset_corr["significant_fdr_0.05"] = dataset_corr["spearman_p_fdr_bh"] < 0.05

dataset_corr.to_csv(OUTPUT_DIR / "r2_vs_cf_dataset_level_correlations_PRIMARY.csv", index=False)

print("PRIMARY DATASET-LEVEL CORRELATIONS")
print(dataset_corr[[
    "metric_label","spearman_rho","spearman_ci95_low","spearman_ci95_high",
    "spearman_p_raw","spearman_p_fdr_bh","significant_fdr_0.05",
    "pearson_r","pearson_p_raw"
]].round(4).to_string(index=False))


In [ ]:
# 10. ROBUSTNESS CHECK: MEAN VS MEDIAN ACROSS METHODS
sensitivity_rows = []
for metric, label in MAIN_METRICS.items():
    rho_mean, p_mean = spearmanr(dataset_mean["forecast_r2"], dataset_mean[metric])
    rho_median, p_median = spearmanr(dataset_median["forecast_r2"], dataset_median[metric])
    sensitivity_rows.append({
        "metric": metric,
        "metric_label": label,
        "spearman_rho_mean_aggregation": rho_mean,
        "spearman_p_mean_aggregation": p_mean,
        "spearman_rho_median_aggregation": rho_median,
        "spearman_p_median_aggregation": p_median,
    })

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUTPUT_DIR / "r2_vs_cf_mean_vs_median_sensitivity.csv", index=False)
print("MEAN VS MEDIAN AGGREGATION SENSITIVITY")
print(sensitivity[["metric_label","spearman_rho_mean_aggregation","spearman_rho_median_aggregation"]].round(3).to_string(index=False))


## Main paper heatmap

This is the clearest figure for the exact meta-analysis question: **Across the 21 datasets, how is forecasting-model quality associated with overall counterfactual quality?**

Each cell is one Spearman correlation computed from 21 dataset-level observations. A star is added only if the association remains significant after Benjamini–Hochberg correction across the seven primary tests.


In [ ]:
# 11. FOCUSED R²-vs-CF SPEARMAN HEATMAP
metric_labels = dataset_corr["metric_label"].tolist()
rho_values = dataset_corr["spearman_rho"].to_numpy()[None, :]
sig_values = dataset_corr["significant_fdr_0.05"].to_numpy()[None, :]

fig, ax = plt.subplots(figsize=(11.5, 2.8))
im = ax.imshow(rho_values, aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(metric_labels)))
ax.set_xticklabels(metric_labels, rotation=30, ha="right")
ax.set_yticks([0])
ax.set_yticklabels(["Forecast $R^2$"])

for j, rho in enumerate(rho_values[0]):
    label = f"{rho:.2f}" + ("*" if sig_values[0, j] else "")
    ax.text(j, 0, label, ha="center", va="center", fontsize=10)

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label("Spearman rho")
ax.set_title("Dataset-level correlation between forecasting quality and counterfactual metrics")
fig.tight_layout()

focused_heatmap_path = OUTPUT_DIR / "heatmap_r2_vs_cf_dataset_level_spearman.png"
fig.savefig(focused_heatmap_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", focused_heatmap_path)


## Full dataset-level Spearman matrix



In [ ]:
# 12. FULL DATASET-LEVEL SPEARMAN CORRELATION MATRIX
heatmap_columns = [
    "forecast_r2","validity_ratio","prefix_validity","max_consecutive_valid_steps",
    "proximity_l2_scaled","proximity_l1_scaled","sparsity_ratio","runtime_sec"
]
pretty_names = {
    "forecast_r2":"$R^2$","validity_ratio":"Validity","prefix_validity":"Prefix validity",
    "max_consecutive_valid_steps":"Max valid steps","proximity_l2_scaled":"$L_2$ proximity",
    "proximity_l1_scaled":"$L_1$ proximity","sparsity_ratio":"Sparsity","runtime_sec":"Runtime"
}

spearman_matrix = dataset_mean[heatmap_columns].corr(method="spearman")
spearman_matrix.to_csv(OUTPUT_DIR / "dataset_level_spearman_correlation_matrix.csv")
labels = [pretty_names[c] for c in spearman_matrix.columns]

fig, ax = plt.subplots(figsize=(9.2,8.2))
im = ax.imshow(spearman_matrix.values, vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=40, ha="right")
ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{spearman_matrix.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Spearman rho")
ax.set_title("Dataset-level Spearman correlation matrix\n(counterfactual metrics averaged across methods)")
fig.tight_layout()
full_heatmap_path = OUTPUT_DIR / "heatmap_dataset_level_spearman_full.png"
fig.savefig(full_heatmap_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", full_heatmap_path)


## Dataset-level scatter plots

These use **one point per dataset**, so they are aligned with the primary correlation analysis. The fitted line is only a visual guide; statistical interpretation comes from the Spearman table.


In [ ]:
# 13. FOUR-PANEL DATASET-LEVEL SCATTER FIGURE
paper_metrics = [
    ("validity_ratio", "Validity", False),
    ("prefix_validity", "Prefix validity", False),
    ("proximity_l2_scaled", "Scaled L2 proximity", False),
    ("runtime_sec", "Runtime per CF call (s)", True),
]

fig, axes = plt.subplots(2,2,figsize=(12.5,9))
axes = axes.flatten()
public_datasets = {"ETTh1","ETTh2","Weather","Electricity"}

for ax, (metric, ylabel, log_y) in zip(axes, paper_metrics):
    x = dataset_mean["forecast_r2"].to_numpy()
    y = dataset_mean[metric].to_numpy()
    ax.scatter(x, y, s=58, alpha=0.8)
    for _, row in dataset_mean.iterrows():
        if row["dataset"] in public_datasets:
            ax.annotate(row["dataset"], (row["forecast_r2"], row[metric]), xytext=(5,4), textcoords="offset points", fontsize=8)
    finite = np.isfinite(x) & np.isfinite(y)
    if finite.sum() >= 3:
        if log_y:
            coef = np.polyfit(x[finite], np.log10(y[finite]), 1)
            x_line = np.linspace(x[finite].min(), x[finite].max(), 100)
            y_line = 10 ** np.polyval(coef, x_line)
        else:
            coef = np.polyfit(x[finite], y[finite], 1)
            x_line = np.linspace(x[finite].min(), x[finite].max(), 100)
            y_line = np.polyval(coef, x_line)
        ax.plot(x_line, y_line, linewidth=1.5, alpha=0.6)
    rho = dataset_corr.loc[dataset_corr["metric"]==metric, "spearman_rho"].iloc[0]
    q = dataset_corr.loc[dataset_corr["metric"]==metric, "spearman_p_fdr_bh"].iloc[0]
    ax.text(0.03,0.95,f"Spearman rho={rho:.2f}\nFDR q={q:.3f}",transform=ax.transAxes,va="top",fontsize=9)
    ax.set_xlabel("Forecast model test $R^2$")
    ax.set_ylabel(ylabel)
    if log_y:
        ax.set_yscale("log")
    ax.grid(True, alpha=0.25)

fig.suptitle("Forecasting model quality vs. dataset-level counterfactual performance", fontsize=14)
fig.tight_layout(rect=[0,0,1,0.96])
four_panel_path = OUTPUT_DIR / "r2_vs_cf_metrics_dataset_level_four_panel.png"
fig.savefig(four_panel_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", four_panel_path)


## Pearson sensitivity analysis

Pearson is not the primary statistic because relationships need not be linear and several metrics are skewed. It is retained only as a secondary check.


In [ ]:
# 14. PEARSON SENSITIVITY MATRIX
pearson_matrix = dataset_mean[heatmap_columns].corr(method="pearson")
pearson_matrix.to_csv(OUTPUT_DIR / "dataset_level_pearson_correlation_matrix.csv")
print("Pearson sensitivity correlations with R²:")
print(dataset_corr[["metric_label","pearson_r","pearson_p_raw"]].round(4).to_string(index=False))


In [ ]:
# 15. EXACT FILES TO SEND BACK FOR PAPER ANALYSIS
important_outputs = [
    OUTPUT_DIR / "r2_vs_cf_dataset_level_correlations_PRIMARY.csv",
    OUTPUT_DIR / "r2_vs_cf_mean_vs_median_sensitivity.csv",
    OUTPUT_DIR / "dataset_level_mean_cf_metrics.csv",
    OUTPUT_DIR / "heatmap_r2_vs_cf_dataset_level_spearman.png",
    OUTPUT_DIR / "heatmap_dataset_level_spearman_full.png",
    OUTPUT_DIR / "r2_vs_cf_metrics_dataset_level_four_panel.png",
]

print("IMPORTANT FILES TO SEND BACK FOR PAPER ANALYSIS:\n")
for p in important_outputs:
    print(p.name)

print("\nPrimary interpretation should be based on r2_vs_cf_dataset_level_correlations_PRIMARY.csv and heatmap_r2_vs_cf_dataset_level_spearman.png.")
